In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
    def __init__(
        self, 
        dim: int, 
        hidden_dim: int = None, 
        multiple_of: int = 256, 
        ffn_dim_multiplier: float = None
    ):
        """
        Args:
            dim: 输入维度 (d_model)
            hidden_dim: 隐层维度。如果为None，则根据 LLaMA 规则自动计算。
            multiple_of: 确保 hidden_dim 是这个数的倍数 (为了硬件计算效率，通常是 256 或 128)。
            ffn_dim_multiplier: 可选的缩放因子 (例如 LLaMA 3 使用了调整过的倍率)。
        """
        super().__init__()
        
        # --- 1. 现代 FFN 的维度计算逻辑 ---
        if hidden_dim is None:
            # 传统的 FFN 通常是 4 * dim
            hidden_dim = 4 * dim
            # SwiGLU 有 3 个权重矩阵，为了保持参数量和旧版 (2个矩阵) 差不多，
            # 通常将隐层维度缩小为原来的 2/3。
            hidden_dim = int(2 * hidden_dim / 3)
            
            # 如果有自定义缩放 (如 LLaMA 3)
            if ffn_dim_multiplier is not None:
                hidden_dim = int(ffn_dim_multiplier * hidden_dim)
            
            # 向上取整到 multiple_of 的倍数 (对齐内存/Tensor Core 友好)
            hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)
        
        self.hidden_dim = hidden_dim

        # --- 2. 定义三个线性层 (通常没有 Bias) ---
        # Gate Projection: 决定激活多少 (配 SiLU)
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        
        # Up Projection: 原始信号变换 (不配激活)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        
        # Down Projection: 投影回原维度
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        # --- 3. SwiGLU 核心逻辑 ---
        # 对应公式: (SiLU(X @ W_gate) * (X @ W_up)) @ W_down
        
        # F.silu(self.gate_proj(x)) -> 激活门控
        # self.up_proj(x)           -> 线性变换信号
        # * -> Element-wise 乘法 (Gating)
        
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [3]:
B, L, D = 2, 128, 4096  # 典型的 LLaMA 7B 尺寸

# 实例化 (自动计算 hidden_dim)
ffn = FeedForward(dim=D)

x = torch.randn(B, L, D)
out = ffn(x)

print(f"Input: {x.shape}")
print(f"Hidden Dim: {ffn.hidden_dim}") 
# 计算: 4096 * 4 * (2/3) ≈ 10922 -> 对齐 256 -> 11008 (LLaMA 7B 的真实参数)
print(f"Output: {out.shape}")

Input: torch.Size([2, 128, 4096])
Hidden Dim: 11008
Output: torch.Size([2, 128, 4096])
